# Recommender systems

Recommender systems are algorithms that provide personalized suggestions for items that are most relevant to each user preferences. Recommender systems are used in a variety of areas, from playlist generators for video or music services, product recommenders for online stores, or content recommenders for social media platforms and c applications like online dating. These systems can use a single type of input, like music, or multiple inputs like news, books, movies, etc.

Recommender systems usually make use of either or both of content\-based filtering and collaborative filtering.

## Content\-based filtering

Content\-based filtering is based on discrete, pre\-tagged characteristics of items to recommend items with similar attributes. Content\-based filtering describe users and items by their known metadata. Each item is represented by a set of relevant tags. For example, valid tags for a movie would be "action", "comedy", or "dramma". Users are represented by a user profile which is created from known user information, such as gender, age, location, etc.

## Collaborative filtering

Collaborative filtering builds a model from users past behavior. This is, items previously viewed, purchased, or rated by users. The model is used to predict items or ratings for items that users may be have interest in. Collaborative filtering do not use metadata, it leverages the feedbacks or activity history of users in order to predict the rating of a user on a given item by inferring dependencies between users and items from the observed activities.



In [42]:
import numpy as np
import pandas as pd

## The Movie Database

The Movie Database \(TMDB\) is a popular, user editable database for movies and TV shows with over 1.5 million contributors. TMDB dataset has two files: 'tmdb\_5000\_movies.csv' provides general information about movies, and 'tmdb\_5000\_credits.csv' provides detailed information about the cast and the crew.

The movies dataset:

- budget
- genres
- homepage
- keywords
- original\_language
- original\_title
- overview
- popularity
- production\_companies
- production\_countries
- release\_date
- revenue
- runtime
- spoken\_languages
- status
- tagline
- title
- vote\_average
- vote\_count

The crew dataset:

- movie\_id
- title
- cast
- crew

The dataset 'tmdb\_movies\_raw.csv' is a subset after merging of the original datasets. It only contains the most relevant information of 5000 movies. This dataset is used to train a recommender system. 

- movie\_id
- title
- overview
- genres
- keywords
- cast
- director



In [43]:
df = pd.read_csv('tmdb_movies_raw.csv')

In [44]:
df.head(3)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4806 entries, 0 to 4805
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4806 non-null   int64 
 1   title     4806 non-null   object
 2   overview  4806 non-null   object
 3   genres    4806 non-null   object
 4   keywords  4806 non-null   object
 5   cast      4806 non-null   object
 6   crew      4806 non-null   object
dtypes: int64(1), object(6)
memory usage: 263.0+ KB


In [46]:
# column title is a string

df.iloc[0]['title']

'Avatar'

In [47]:
# column overview is a string


In [48]:
# column genres, keyword, cast, and crew are dictionaries


In [49]:
# convert a dictionary into a list

import ast

def dictionary_to_list(d):
    output = []

    for item in ast.literal_eval(d):
        output.append(item['name'])

    return output

In [50]:
# columns genres and keywords are converted into lists

df['genres'] = df['genres'].apply(dictionary_to_list)
df['keywords'] = df['keywords'].apply(dictionary_to_list)

In [51]:
# column genres is a list


In [52]:
# column keywords is a list


In [53]:
# extract the 3 main actors from the cast

def extract_main_actors_from_cast(d):
    output = []

    counter = 0

    for item in ast.literal_eval(d):
        if counter < 3:
            output.append(item['name'])

        counter+=1

    return output

In [54]:
df['cast'] = df['cast'].apply(extract_main_actors_from_cast)

In [55]:
df.iloc[0]['cast']

['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']

In [56]:
# column crew contains the movie director

df.iloc[0]['crew']

'[{"credit_id": "52fe48009251416c750aca23", "department": "Editing", "gender": 0, "id": 1721, "job": "Editor", "name": "Stephen E. Rivkin"}, {"credit_id": "539c47ecc3a36810e3001f87", "department": "Art", "gender": 2, "id": 496, "job": "Production Design", "name": "Rick Carter"}, {"credit_id": "54491c89c3a3680fb4001cf7", "department": "Sound", "gender": 0, "id": 900, "job": "Sound Designer", "name": "Christopher Boyes"}, {"credit_id": "54491cb70e0a267480001bd0", "department": "Sound", "gender": 0, "id": 900, "job": "Supervising Sound Editor", "name": "Christopher Boyes"}, {"credit_id": "539c4a4cc3a36810c9002101", "department": "Production", "gender": 1, "id": 1262, "job": "Casting", "name": "Mali Finn"}, {"credit_id": "5544ee3b925141499f0008fc", "department": "Sound", "gender": 2, "id": 1729, "job": "Original Music Composer", "name": "James Horner"}, {"credit_id": "52fe48009251416c750ac9c3", "department": "Directing", "gender": 2, "id": 2710, "job": "Director", "name": "James Cameron"},

In [57]:
# extract the name of the movie director from the cast

def extract_director(d):
    output = []

    for item in ast.literal_eval(d):
        if item['job'] == 'Director':
            output.append(item['name'])

            break

    return output

In [58]:
# extract the movie director from crew and rename the column as director

df['crew'] = df['crew'].apply(extract_director)
df.rename(columns={'crew': 'director'}, inplace=True)

In [59]:
df.iloc[0]['director']

['James Cameron']

In [60]:
# column overview is converted into a list of words

df['overview'] = df['overview'].apply(lambda x:x.split())

In [61]:
# overview is a list of words

df.iloc[0]['overview']

['In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to',
 'the',
 'moon',
 'Pandora',
 'on',
 'a',
 'unique',
 'mission,',
 'but',
 'becomes',
 'torn',
 'between',
 'following',
 'orders',
 'and',
 'protecting',
 'an',
 'alien',
 'civilization.']

In [62]:
df.head(3)

,movie_id,title,overview,genres,keywords,cast,director
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]


In [63]:
# remove spaces between words: for example "Science Fiction" will be "ScienceFiction"

def trim_string(list_of_words):
    output = []

    for item in list_of_words:
        output.append(item.replace(" ",""))

    return output

In [64]:
trim_string(df.iloc[0]['genres'])

['Action', 'Adventure', 'Fantasy', 'ScienceFiction']

In [65]:
# remove spaces from genres, keywords, cast, and director

df['genres'] = df['genres'].apply(trim_string)
df['keywords'] = df['keywords'].apply(trim_string)
df['cast'] = df['cast'].apply(trim_string)
df['director'] = df['director'].apply(trim_string)

In [66]:
# extract a new feature concatenating overview, genres, keywords, cast, and director

df['attributes'] = df['overview'] + df['genres'] + df['keywords'] + df['cast'] + df['director']

In [67]:
df.head(3)

,movie_id,title,overview,genres,keywords,cast,director,attributes
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],"[A, cryptic, message, from, Bond’s, past, send..."


In [68]:
# keep columns title, and attributes

df = df[['movie_id','title','attributes']]

In [69]:
# the column attributes is a list of strings

df.iloc[0]['attributes']

['In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to',
 'the',
 'moon',
 'Pandora',
 'on',
 'a',
 'unique',
 'mission,',
 'but',
 'becomes',
 'torn',
 'between',
 'following',
 'orders',
 'and',
 'protecting',
 'an',
 'alien',
 'civilization.',
 'Action',
 'Adventure',
 'Fantasy',
 'ScienceFiction',
 'cultureclash',
 'future',
 'spacewar',
 'spacecolony',
 'society',
 'spacetravel',
 'futuristic',
 'romance',
 'space',
 'alien',
 'tribe',
 'alienplanet',
 'cgi',
 'marine',
 'soldier',
 'battle',
 'loveaffair',
 'antiwar',
 'powerrelations',
 'mindandsoul',
 '3d',
 'SamWorthington',
 'ZoeSaldana',
 'SigourneyWeaver',
 'JamesCameron']

In [81]:
df.head(3)

,movie_id,title,attributes
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...


In [71]:
# convert the contents of the column attributes into a string

df['attributes'] = df['attributes'].apply(lambda x: " ".join(x))

In [72]:
df.iloc[0]['attributes']

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'

In [73]:
# convert attributes to lower case

df['attributes'] = df['attributes'].apply(lambda x:x.lower())

In [74]:
df.iloc[0]['attributes']

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron'

# Stemming

Stemming is a text normalization technique that reduces inflected forms of a word into one morphological lexeme. This root\-form lexeme is called "stem" of "lemma" in linguistics. For example, words "invest", "investing", "investment", "investments" are reduced to the root form "invest". By reducing derivational word forms to one lexeme, recommender systems equate morphologically related words.

The words in the field 'attributes' are reduced to their roots by removing suffixes and prefixes. The stemmer looks for a list of common suffixes and prefixes and removes them.



In [75]:
from nltk.stem import PorterStemmer

In [76]:
ps = PorterStemmer()

def stem_string(string):
    output = []

    for word in string.split():
        output.append(ps.stem(word))

    return " ".join(output)

In [77]:
df['attributes'] = df['attributes'].apply(stem_string)

In [78]:
df.iloc[0]['attributes']

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d samworthington zoesaldana sigourneyweav jamescameron'

In [79]:
# save the dataframe after preprocessing

df.to_csv('tmdb_movies.csv', index=False)